# Práctica 2 — Titanic Dataset

## Resultados de aprendizaje
Al finalizar la práctica, serás capaz de:

1. Distinguir los datos de un registro individual de la información que se obtiene al resumir varios registros. Por ejemplo, identificar la diferencia entre la clase del boleto de un pasajero y el total de pasajeros por clase.
2. Generar reportes descriptivos en Python a partir de un archivo CSV, agrupando registros y calculando conteos, totales y otras medidas de resumen.
3. Verificar la consistencia de un reporte comparando sus subtotales con el total del archivo y documentando cómo se trataron los valores faltantes.
4. Interpretar los resultados y comunicar por escrito qué muestran sobre los pasajeros incluidos en el archivo.

## Introducción

En 1912, el personal encargado del embarque del Titanic utilizaba listas en papel para registrar a los pasajeros. En esta práctica, estudiaremos ese registro como un ejemplo de procesamiento manual de transacciones. Las listas cumplían una función operativa: dejar constancia de quiénes viajaban. El dataset reúne datos de los pasajeros, como su clase y puerto de embarque, e incorpora un dato conocido después del naufragio: si sobrevivieron.

### Diccionario de datos

El archivo `titanic.csv` contiene **891 registros y 12 columnas**. Cada fila representa a un pasajero.

| Columna | Tipo de dato | Descripción |
|---|---|---|
| `PassengerId` | Identificador entero | Código único asignado a cada pasajero en el dataset. |
| `Survived` | Categoría binaria | Resultado registrado después del naufragio. `0` = falleció, `1` = sobrevivió. |
| `Pclass` | Categoría ordinal | Clase del boleto. `1` = primera, `2` = segunda, `3` = tercera. |
| `Name` | Texto | Nombre del pasajero tal como aparece en el archivo. |
| `Sex` | Categoría | Sexo registrado. Los valores del archivo son `female` y `male`. |
| `Age` | Número | Edad en años. Puede incluir valores fraccionarios, por ejemplo, para bebés menores de un año. |
| `SibSp` | Conteo entero | Número de hermanos, hermanas y cónyuges del pasajero a bordo. |
| `Parch` | Conteo entero | Número de padres, madres, hijos e hijas del pasajero a bordo. |
| `Ticket` | Código de texto | Código del boleto. Puede incluir letras, números y signos, y repetirse en los registros de varios pasajeros. |
| `Fare` | Número decimal | Tarifa registrada en la fila del pasajero. |
| `Cabin` | Código de texto | Identificación de la cabina, cuando figura en el registro. |
| `Embarked` | Categoría | Puerto de embarque. `C` = Cherburgo, `Q` = Queenstown, `S` = Southampton. |


## Preguntas de Interés

1. ¿Cuántos pasajeros hay en el dataset y cuántos PassengerId distintos aparecen? ¿Coinciden los dos resultados?
2. ¿Cuántos pasajeros de cada clase aparecen en el archivo? ¿Qué clase tiene más pasajeros registrados?
3. ¿Cuántos pasajeros embarcaron en cada puerto y cuántos tienen el puerto sin registrar?
4. ¿Cuál es la tarifa promedio, mínima y máxima registrada para cada clase?
5. ¿Cuántos pasajeros de cada clase embarcaron en cada puerto?
6. ¿Cuántos pasajeros sobrevivieron y cuántos fallecieron en cada clase?
7. ¿Cuántos pasajeros viajaban con cero, uno o dos o más familiares registrados a bordo?
8. ¿Cuántos pasajeros tenían menos de 18 años, cuántos tenían 18 o más y cuántos tenían la edad sin registrar? Desglose el resultado por clase.
9. ¿Cuántos códigos de boleto distintos hay? ¿Cuántos aparecen asociados a más de un pasajero y cuál aparece en más registros?
10. ¿En qué clase hay más registros sin cabina? Para cada clase, muestren cuántos registros tienen cabina, cuántos carecen de ella y cuántos hay en total.

## Ejercicio:

Por cada una de las preguntas de interés, crea un bloque de código Python para escribir el código que permita obtener la información que responda a la pregunta.

Presenta los resultados en tablas con encabezados claros e indica cómo trataste los valores faltantes cuando afecten un cálculo. Todos los resultados deben referirse a los pasajeros del archivo.

### Ejercicio 1

1. ¿Cuántos pasajeros hay en el dataset y cuántos PassengerId distintos aparecen? ¿Coinciden los dos resultados?

In [ ]:
import pandas as pd

# Cargar el archivo CSV en un DataFrame
df = pd.read_csv('titanic.csv')

In [ ]:
total_pasajeros = len(df)
ids_distintos = df["PassengerId"].nunique() # .nunique() devuelve el número de valores únicos

# Crear una tabla con los resultados
tabla1 = pd.DataFrame({
    "Métrica": ["Pasajeros (filas) en el dataset",
                "PassengerId distintos",
                "¿Coinciden ambos resultados?"],
    "Resultado": [total_pasajeros, ids_distintos,
                  "Sí" if total_pasajeros == ids_distintos else "No"]
})

# Muestra la tabla estilo HTML
display(tabla1)

__Interpretación y consistencia:__ Hay 891 pasajeros registrados y 891 códigos PassengerId distintos. Ambos valores coinciden, lo que confirma que cada fila representa un pasajero único sin registros duplicados por identificador.

### Ejercicio 2

2. ¿Cuántos pasajeros de cada clase aparecen en el archivo? ¿Qué clase tiene más pasajeros registrados?

In [ ]:
clases = {1: "Primera", 2: "Segunda", 3: "Tercera"} # Mapeo de clases
orden_clases = list(clases.values())

# Mapear la columna "Pclass" a nombres de clases y contar los pasajeros por clase
tabla2 = (df["Pclass"].map(clases).value_counts()
          .reindex(orden_clases)
          .rename_axis("Clase")
          .reset_index(name = "Pasajeros"))

# Sumar el total de pasajeros y agregarlo como una fila al final de la tabla
tabla2.loc[len(tabla2)] = ["Total", tabla2["Pasajeros"].sum()]
display(tabla2)

# Ordena la tabla por el número de pasajeros en orden descendente y
# seleccionar la primera fila (excluyendo la fila de total)
clase_max = tabla2.iloc[:-1].sort_values("Pasajeros", ascending=False).iloc[0]
print(f"Clase con más pasajeros registrados: {clase_max['Clase']} ({clase_max['Pasajeros']} pasajeros)")

__Interpretación y tratamiento de faltantes:__ La columna Pclass no presenta valores faltantes (la suma de subtotales es 891). La tercera clase tiene la mayor cantidad de pasajeros registrados en el archivo.

### Ejercicio 3

3. ¿Cuántos pasajeros embarcaron en cada puerto y cuántos tienen el puerto sin registrar?

In [ ]:
puertos = {"C": "Cherburgo (C)", "Q": "Queenstown (Q)", "S": "Southampton (S)"}
orden_puertos = list(puertos.values()) + ["Sin registrar"]

# Los valores faltantes se muestran como una categoría propia ("Sin registrar")
puerto = df["Embarked"].map(puertos).fillna("Sin registrar")

tabla3 = (puerto.value_counts()
          .reindex(orden_puertos, fill_value=0)
          .rename_axis("Puerto de embarque")
          .reset_index(name="Pasajeros"))
tabla3.loc[len(tabla3)] = ["Total", tabla3["Pasajeros"].sum()]
display(tabla3)

print("Pasajeros con puerto sin registrar:", df["Embarked"].isna().sum())

__Tratamiento de valores faltantes:__ Se colocó temporalmente la etiqueta 'Sin registrar' sobre los valores nulos (NaN) de la columna Embarked para incluirlos explícitamente en el reporte y garantizar que el total sume 891 pasajeros.

### Ejercicio 4

4. ¿Cuál es la tarifa promedio, mínima y máxima registrada para cada clase?

In [ ]:
clases = {1: "Primera", 2: "Segunda", 3: "Tercera"}
orden_clases = list(clases.values())

# .groupby, agrupa por clase y calcula la tarifa promedio, mínima y máxima
tabla4 = (df.groupby(df["Pclass"].map(clases))["Fare"]
          .agg(["mean", "min", "max"])
          .reindex(orden_clases)
          .round(2)
          .rename(columns={"mean": "Tarifa promedio",
                           "min": "Tarifa mínima",
                           "max": "Tarifa máxima"})
          .rename_axis("Clase")
          .reset_index())
display(tabla4)


__Tratamiento de valores faltantes:__ La columna Fare cuenta con los 891 valores completos (sin nulos). Se observa que existen registros con tarifa 0.00 en las tres clases (correspondientes a boletos de cortesía o tripulación/trabajadores embarcados como pasajeros).

### Ejercicio 5

5. ¿Cuántos pasajeros de cada clase embarcaron en cada puerto?

In [ ]:
clases = {1: "Primera", 2: "Segunda", 3: "Tercera"}
puertos = {"C": "Cherburgo (C)", "Q": "Queenstown (Q)", "S": "Southampton (S)"}
orden_clases = list(clases.values())
orden_puertos = list(puertos.values()) + ["Sin registrar"]

clase = df["Pclass"].map(clases).rename("Clase")

# Los puertos sin registrar se conservan como columna para que los subtotales cuadren
puerto = df["Embarked"].map(puertos).fillna("Sin registrar").rename("Puerto")

# crosstab genera una tabla de contingencia con los conteos de pasajeros por clase y puerto
tabla5 = pd.crosstab(clase, puerto, margins=True, margins_name="Total")
tabla5 = tabla5.reindex(index=orden_clases + ["Total"],
                        columns=orden_puertos + ["Total"],
                        fill_value=0)
tabla5.index.name = "Clase"
tabla5.columns.name = "Puerto"
display(tabla5)

__Tratamiento de valores faltantes:__ Al reemplazar los valores faltantes de Embarked por 'Sin registrar (NaN)' antes de cruzar las variables, los 2 pasajeros sin puerto aparecen contabilizados en Primera Clase, logrando que la tabla cruzada cuadre exactamente con los 891 registros.

### Ejercicio 6

6. ¿Cuántos pasajeros sobrevivieron y cuántos fallecieron en cada clase?

In [ ]:
clases = {1: "Primera", 2: "Segunda", 3: "Tercera"}
orden_clases = list(clases.values())

clase = df["Pclass"].map(clases).rename("Clase")
resultado = df["Survived"].map({0: "Fallecieron", 1: "Sobrevivieron"}).rename("Resultado")

tabla6 = pd.crosstab(clase, resultado, margins=True, margins_name="Total")
tabla6 = tabla6.reindex(index=orden_clases + ["Total"],
                        columns=["Sobrevivieron", "Fallecieron", "Total"],
                        fill_value=0)
tabla6.index.name = "Clase"
tabla6.columns.name = "Resultado"
display(tabla6)

__Interpretación y consistencia:__ No hay valores faltantes en Survived ni en Pclass. La tabla muestra que Primera Clase fue la única donde sobrevivieron más pasajeros de los que fallecieron, mientras que Tercera Clase concentró la mayor cantidad de fallecidos.

### Ejercicio 7

7. ¿Cuántos pasajeros viajaban con cero, uno o dos o más familiares registrados a bordo?

In [ ]:
# numpy biblioteca para trabajar con arreglos y funciones matemáticas
import numpy as np

# Familiares a bordo = hermanos/cónyuges (SibSp) + padres/hijos (Parch)
familiares = df["SibSp"] + df["Parch"]
etiquetas = ["0 familiares", "1 familiar", "2 o más familiares"]

# .cut divide los datos en intervalos y asigna etiquetas a cada intervalo
grupo_familiar = pd.cut(familiares, bins=[-1, 0, 1, np.inf], labels=etiquetas)

tabla7 = (grupo_familiar.value_counts(dropna=False)
          .reindex(etiquetas)
          .rename_axis("Familiares registrados a bordo")
          .reset_index(name="Pasajeros"))
tabla7.loc[len(tabla7)] = ["Total", tabla7["Pasajeros"].sum()]
display(tabla7)

__Tratamiento de valores faltantes:__ Tanto SibSp como Parch están completos para los 891 pasajeros. La suma de ambas columnas permite reconstruir el tamaño del núcleo familiar directo a bordo sin pérdida de datos.

### Ejercicio 8

8. ¿Cuántos pasajeros tenían menos de 18 años, cuántos tenían 18 o más y cuántos tenían la edad sin registrar? Desglose el resultado por clase.

In [ ]:
import numpy as np

clases = {1: "Primera", 2: "Segunda", 3: "Tercera"}
orden_clases = list(clases.values())
orden_edad = ["Menores de 18", "18 o más", "Edad sin registrar"]

clase = df["Pclass"].map(clases).rename("Clase")

# La edad faltante se evalúa primero para que no se cuente como menor ni como adulto
grupo_edad = pd.Series(
    np.select([df["Age"].isna(), df["Age"] < 18, df["Age"] >= 18],
              ["Edad sin registrar", "Menores de 18", "18 o más"],
              default="Edad sin registrar"),
    index=df.index, name="Grupo de edad")

tabla8 = pd.crosstab(clase, grupo_edad, margins=True, margins_name="Total")
tabla8 = tabla8.reindex(index=orden_clases + ["Total"],
                        columns=orden_edad + ["Total"],
                        fill_value=0)
tabla8.index.name = "Clase"
tabla8.columns.name = "Grupo de edad"
display(tabla8)

print(tabla8.loc["Total", orden_edad].to_string())

__Tratamiento de valores faltantes:__ La columna Age presenta 177 valores faltantes. En lugar de eliminarlos o imputarlos con promedios (lo que distorsionaría el conteo real de menores y adultos), se agruparon en la categoría explícita 'Sin registrar (NaN)' para mantener la integridad de los 891 pasajeros.

### Ejercicio 9

9. ¿Cuántos códigos de boleto distintos hay? ¿Cuántos aparecen asociados a más de un pasajero y cuál aparece en más registros?

In [ ]:
conteo_boletos = df["Ticket"].value_counts()

boletos_distintos = df["Ticket"].nunique()
boletos_compartidos = (conteo_boletos > 1).sum()
max_registros = conteo_boletos.max()
boletos_max = conteo_boletos[conteo_boletos == max_registros].index.tolist()

tabla9 = pd.DataFrame({
    "Métrica": ["Códigos de boleto distintos",
                "Códigos asociados a más de un pasajero",
                "Código con más registros",
                "Número de registros de ese código"],
    "Resultado": [boletos_distintos,
                  boletos_compartidos,
                  ", ".join(map(str, boletos_max)),
                  max_registros]
})
display(tabla9)

__Interpretación y consistencia:__ La columna Ticket no tiene valores nulos. Existen 681 códigos distintos para 891 pasajeros debido a que familias o grupos compartían un mismo código de boleto (por ejemplo, los boletos 347082, CA. 2343 y 1601 aparecen en 7 registros cada uno).

### Ejercicio 10

10. ¿En qué clase hay más registros sin cabina? Para cada clase, muestren cuántos registros tienen cabina, cuántos carecen de ella y cuántos hay en total.

In [ ]:
import numpy as np

clases = {1: "Primera", 2: "Segunda", 3: "Tercera"}
orden_clases = list(clases.values())

clase = df["Pclass"].map(clases).rename("Clase")

# Se considera "sin cabina" tanto el valor vacío (NaN) como el texto en blanco
sin_cabina = df["Cabin"].isna() | (df["Cabin"].astype(str).str.strip() == "")
estado_cabina = pd.Series(np.where(sin_cabina, "Sin cabina", "Con cabina"),
                          index=df.index, name="Estado de cabina")

tabla10 = pd.crosstab(clase, estado_cabina, margins=True, margins_name="Total")
tabla10 = tabla10.reindex(index=orden_clases + ["Total"],
                          columns=["Con cabina", "Sin cabina", "Total"],
                          fill_value=0)
tabla10.index.name = "Clase"
tabla10.columns.name = None
display(tabla10)

clase_mas_sin_cabina = tabla10.loc[orden_clases, "Sin cabina"].idxmax()
print(f"Clase con más registros sin cabina: {clase_mas_sin_cabina} "
      f"({tabla10.loc[clase_mas_sin_cabina, 'Sin cabina']} registros)")

__Tratamiento de valores faltantes:__ La ausencia de dato (NaN) en Cabin no representa un error de cálculo sino un registro no asignado o no documentado en las listas. Se transformó la columna en una variable categórica binaria (Con cabina / Sin cabina) para cuantificar la completitud del dato por clase, evidenciando que Tercera Clase concentra la mayor cantidad de registros sin cabina (479 de 491 pasajeros).